# Minimum viable product
In this notebook I'll try to see if Meetings Capture intended functionality could actually work. 

In [1]:
import imghdr
import json
import os
from itertools import chain

import google.generativeai as genai
from google.generativeai.generative_models import GenerativeModel
from google.generativeai.types import GenerateContentResponse
from pandas import DataFrame
from PIL import Image
from PIL.ImageFile import ImageFile
from PIL.JpegImagePlugin import JpegImageFile
from pyconfparser import ConfigFactory

C:\Users\Admin\AppData\Local\Temp\ipykernel_4772\4264719874.py:2: DeprecationWarning: 'imghdr' is deprecated and slated for removal in Python 3.13
  import imghdr
c:\Users\Admin\Desktop\Repos\MeetingsCapture\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Helper functions

In [2]:
def __open_img_with_PIL(img_path: str) -> ImageFile:
    """Opens an image file using the Python Imaging Library (PIL).

    This function loads an image from the specified file path and returns it as
    a PIL Image object.

    Args:
        img_path (str): The file path of the image to be opened.

    Returns:
        ImageFile.ImageFile: The opened image as a PIL Image object.
    """
    return Image.open(img_path)


def get_imgs_in_dir(dir_path: str) -> list[ImageFile]:
    """Finds and loads all valid images from a directory.

    This function scans the specified directory, detects valid image files,
    and loads them as PIL Image objects.

    Args:
        dir_path (str): The path to the directory containing image files.

    Returns:
        list[ImageFile.ImageFile]: A list of loaded PIL Image objects.

    Raises:
        AssertionError: If the given directory path does not exist.
    """
    assert os.path.exists(dir_path), "The given path does not exist!"

    found_imgs: list[ImageFile] = []
    for file in os.listdir(dir_path):
        absolute_path: str = os.path.join(dir_path, file)

        if os.path.isfile(absolute_path) and imghdr.what(absolute_path):
            PIL_img: ImageFile = __open_img_with_PIL(absolute_path)
            found_imgs.append(PIL_img)

    return found_imgs


def extract_raw_json(unparsed_json: str) -> str:
    """Extracts the pure JSON content.

    Args:
        unparsed_json (str): A JSON string potentially wrapped in Markdown-style
                             triple backticks.

    Returns:
        str: The cleaned JSON string without Markdown formatting.
    """
    return unparsed_json.replace("```json\n", "").replace("\n```", "")


def flatten_list(array: list) -> list:
    return list(chain(*array))

In [ ]:
def make_request_2_model(
    model: GenerativeModel, prompt: str, images: JpegImageFile | list[JpegImageFile]
) -> list[dict]:
    """Makes a request to a generative model to structure image content into JSON format.

    Args:
        model (GenerativeModel): Generative model to make the request to.
        prompt (str): Prompt to be used in the request.
        images (JpegImageFile | list[JpegImageFile]):
        Image or list of images to be used in the request.

    Returns:
        list[dict]: Structured images content in JSON format.
    """
    images: list[JpegImageFile] = list(images) if not isinstance(images, list) else images

    all_imgs_content: list[dict] = []
    for img in images:
        response: GenerateContentResponse = model.generate_content([prompt, img], stream=True)
        response.resolve()  # wait for the response to be ready
        parsed_json = extract_raw_json(response.text)
        all_imgs_content.append(json.loads(parsed_json))
    return flatten_list(all_imgs_content)

## Main functionality

In [4]:
conf = ConfigFactory.get_conf(r"conf/config.json")
genai.configure(api_key=conf.gemini_api_key)
MODEL = genai.GenerativeModel(conf.model_type)

imgs: list[ImageFile] = get_imgs_in_dir(r"../img/2024")
imgs_structured_content: list[dict] = make_request_2_model(
    model=MODEL, prompt=conf.prompt, images=imgs
)
df: DataFrame = DataFrame(imgs_structured_content)
df.to_excel(conf.file_path + "reuniones_2024.xlsx")  # TODO: improve this with filepath join